# 🎯 [MOV02_M05] 수송선단 생존 확률 및 정시 도착 신뢰도 점수 산출 엔진 개발
### Google Colab 실습 환경 (수송·물류·창고 / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/MOV02/mov02_m05_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 수송·물류·창고 (Level 2) |
| **미션 ID** | MOV02_M05 |
| **미션 주제** | 수송선단 생존 확률 및 정시 도착 신뢰도 점수 산출 엔진 개발 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
NATO 동맹전환사령부(ACT)는 다국적 군수 지원에서 수송 요청부터 도착까지의 시간, 처리 기간, 물동량과 수송능력, 지연 원인을 한 화면에서 보는 물류·수송 성과 가시화를 강조해 왔다. 본 프로젝트는 이 공개 사례를 바탕으로 교육용 가상 도로망과 수송 자산 자료를 코드로 만들어, 수송지원 실적과 비용을 근거로 경로와 자산을 결정하는 과정을 구현한다. 학습자는 자산·도로망 스키마 정의, 도로 파괴·통제 구역 전처리, 다익스트라 Baseline, 위협 회피·적재 효율 추천, 생존확률·정시도착 신뢰도 점수, 관제 대시보드, 수송통제관 승인(HITL), 도로 50곳 차단 스트레스 시험, AAR 재보정, 최종 통합을 차례로 수행한다. 최종 목표는 수송 리드타임과 톤km당 비용을 낮추면서 위험 경로의 강행 여부는 반드시 사람이 결정하도록 하는 수송지원 실적·비용 가시화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 05. 수송선단 생존 확률 및 정시 도착 신뢰도 점수 산출 엔진 개발
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

수송선단 생존 확률 및 정시 도착 신뢰도 점수 산출 엔진을 개발한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **후보 경로마다 수송선단 생존확률과 정시 도착 신뢰도를 계산해 0~100점 신뢰도 점수와 등급을 산출하는 엔진을 개발한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **위협 구간 통과 수로 생존확률을, 도로 등급·교량 대기·위협 구역 감속을 반영한 몬테카를로 모의로 정시 도착 확률을 구 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/ui/mov02_app.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무자 친화적 디자인 및 조작 편의성

#### 미션 PASS 조건

- 정량 기준: 핵심 화면 4종(입력-분석-검토-결과) 구현 완료


- 진행 기준: UI 프로토타입 로컬 서버 구동 및 전체 페이지 네비게이션 확인


- 검증 방법: 사용성 평가(UX Walkthrough) 및 인터랙션 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""후보 수송 경로마다 수송선단 생존 확률(위협 구역 통과 기반)과 정시 도착 신뢰도(몬테카를로 소요시간 모의)를 계산해 0~100점 신뢰도 점수와 등급을 산출하는 엔진을 만들고, 입력-분석-검토-결과 4개 화면으로 표시하며 예측 확률의 보정 정확도를 검증한다."""
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 도로망·수송 자산 생성기 (실제 지명·부대·장비 제원 없음)
# 격자형 가상 도로망(노드=교차점, 간선=도로 구간)에 도로 등급·교량 하중·가상 위협 구역을 붙인다.
# ---------------------------------------------------------------------------
ASSETS = {  # 교육용 가정 제원(실제 장비 제원 아님)
    "truck": {"name": "중형 트럭", "cap_t": 5.0, "speed_kmh": 50, "cost_km": 1.0, "weight_t": 10.0},
    "trailer": {"name": "대형 트레일러", "cap_t": 20.0, "speed_kmh": 40, "cost_km": 2.4, "weight_t": 30.0},
    "airlift": {"name": "수송기", "cap_t": 12.0, "speed_kmh": 300, "cost_km": 9.0, "weight_t": 0.0},
}
ROAD_FACTOR = {"paved": 1.0, "unpaved": 0.6}   # 도로 등급별 속도 계수


def make_network(seed=5, rows=6, cols=8, spacing=5.0):
    """격자형 가상 도로망. 노드 좌표(km)와 간선 목록(거리·도로등급·교량 하중)을 돌려준다."""
    rng = np.random.default_rng(seed)
    nodes = {f"N{r:02d}{c:02d}": (c * spacing + rng.uniform(-1, 1), r * spacing + rng.uniform(-1, 1))
             for r in range(rows) for c in range(cols)}
    edges = []

    def add(a, b):
        (x1, y1), (x2, y2) = nodes[a], nodes[b]
        bridge = bool(rng.random() < 0.15)
        edges.append({"id": f"E{len(edges):03d}", "u": a, "v": b,
                      "km": round(float(np.hypot(x2 - x1, y2 - y1) * rng.uniform(1.0, 1.25)), 2),
                      "road": "paved" if rng.random() < 0.65 else "unpaved", "bridge": bridge,
                      "limit_t": float(rng.choice([20, 40, 60])) if bridge else None, "open": True})
    for r in range(rows):
        for c in range(cols):
            if c + 1 < cols:
                add(f"N{r:02d}{c:02d}", f"N{r:02d}{c + 1:02d}")
            if r + 1 < rows:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c:02d}")
            if r + 1 < rows and c + 1 < cols and rng.random() < 0.25:
                add(f"N{r:02d}{c:02d}", f"N{r + 1:02d}{c + 1:02d}")
    return nodes, edges


def make_threats(seed=9, n=3, xlim=(8, 28), ylim=(3, 22)):
    """가상 위협 구역(원형). p = 구간 1회 통과 시 피해 가능성(교육용 가정값)."""
    rng = np.random.default_rng(seed)
    return [{"id": f"TZ{i + 1}", "x": float(rng.uniform(*xlim)), "y": float(rng.uniform(*ylim)),
             "r": float(rng.uniform(3, 5)), "p": float(round(rng.uniform(0.05, 0.2), 3))} for i in range(n)]


def edge_risk(e, nodes, threats):
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    s = np.linspace(0, 1, 6)
    risk = 0.0
    for t in threats:
        if np.min(np.hypot(x1 + (x2 - x1) * s - t["x"], y1 + (y2 - y1) * s - t["y"])) < t["r"]:
            risk = max(risk, t["p"])
    return risk


def edge_minutes(e, asset="truck"):
    return e["km"] / (ASSETS[asset]["speed_kmh"] * ROAD_FACTOR[e["road"]]) * 60


def passable(e, gross_t):
    return e["open"] and (not e["bridge"] or e["limit_t"] >= gross_t)


def build_adj(nodes, edges, weight_fn):
    """weight_fn(e)가 None이면 통행 불가 간선으로 보고 제외한다."""
    adj = {n: [] for n in nodes}
    for e in edges:
        w = weight_fn(e)
        if w is not None:
            adj[e["u"]].append((e["v"], w, e))
            adj[e["v"]].append((e["u"], w, e))
    return adj


def dijkstra(adj, src, dst):
    """다익스트라 최단경로. (비용, 노드 경로, 간선 목록). 도달 불가면 (inf, [], [])."""
    dist, prev, done, pq = {src: 0.0}, {}, set(), [(0.0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if u in done:
            continue
        done.add(u)
        if u == dst:
            break
        for v, w, e in adj[u]:
            if d + w < dist.get(v, float("inf")):
                dist[v], prev[v] = d + w, (u, e)
                heapq.heappush(pq, (d + w, v))
    if dst not in done:
        return float("inf"), [], []
    path, es = [dst], []
    while path[-1] != src:
        u, e = prev[path[-1]]
        es.append(e); path.append(u)
    return dist[dst], path[::-1], es[::-1]
# ---------------------------------------------------------------------------

nodes, edges = make_network()
threats = make_threats()
DEPOT = "N0000"
for e in edges:
    if e["id"] in ("E017", "E040", "E046", "E071"):
        e["open"] = False
    e["risk"] = edge_risk(e, nodes, threats)
SIGMA = {"paved": 0.12, "unpaved": 0.30}      # 도로 등급별 소요시간 변동(로그정규 표준편차, 가정값)


def candidates(dest, lams=(0.0, 0.5, 1.5, 3.0, 0.0, 0.5), k=4):
    """위험 회피 강도를 바꾸고 이미 쓴 구간에 벌점을 주어 서로 다른 후보 경로를 최대 k개 만든다."""
    out, seen, pen = [], set(), {}
    for lam in lams:
        adj = build_adj(nodes, edges, lambda e: (edge_minutes(e) + lam * e["risk"] * 100) * pen.get(e["id"], 1.0) if passable(e, 15) else None)
        _, path, es = dijkstra(adj, DEPOT, dest)
        for e in es:
            pen[e["id"]] = pen.get(e["id"], 1.0) * 1.3
        if path and tuple(path) not in seen and len(out) < k:
            seen.add(tuple(path)); out.append({"path": path, "es": es})
    return out


def simulate_minutes(es, n, rng):
    """경로 소요시간 몬테카를로: 도로 등급별 변동 + 교량 통과 대기 + 위협 구역 감속."""
    total = np.zeros(n)
    for e in es:
        total += edge_minutes(e) * rng.lognormal(0, SIGMA[e["road"]], n)
        if e["bridge"]:
            total += rng.exponential(3, n)
        if e["risk"] > 0:
            total += rng.uniform(0, 10, n)
    return total


def score_route(c, deadline_min, n=3000, seed=0):
    surv = float(np.prod([1 - e["risk"] for e in c["es"]]))
    sims = simulate_minutes(c["es"], n, np.random.default_rng(seed))
    ontime = float((sims <= deadline_min).mean())
    s = 100 * surv * ontime
    return {"생존확률": surv, "정시도착": ontime, "예상소요(P50)": float(np.median(sims)), "예상소요(P90)": float(np.percentile(sims, 90)),
            "점수": round(s, 1), "등급": "신뢰" if s >= 85 else ("주의" if s >= 60 else "위험"), "sims": sims}


# ---------------- 화면 1. 입력 ----------------
REQ = {"req_id": "REQ-0101", "dest": "N0307", "cargo_t": 12.0, "deadline_min": 80}
print("[화면 1: 입력] 수송 요청", {k: v for k, v in REQ.items()}, "| 가상 위협 구역", len(threats), "개")

# ---------------- 화면 2. 분석 ----------------
cands = candidates(REQ["dest"])
rows = []
for i, c in enumerate(cands):
    r = score_route(c, REQ["deadline_min"], seed=i)
    c.update(r)
    rows.append({"경로": f"R{i + 1}", "구간 수": len(c["es"]), "위협 통과": sum(e["risk"] > 0 for e in c["es"]),
                 **{k: r[k] for k in ("생존확률", "정시도착", "예상소요(P50)", "예상소요(P90)", "점수", "등급")}})
table = pd.DataFrame(rows)
fmt = {"생존확률": "{:.1%}", "정시도착": "{:.1%}", "예상소요(P50)": "{:.0f}분", "예상소요(P90)": "{:.0f}분"}
show = table.copy()
for k, f in fmt.items():
    show[k] = show[k].map(f.format)
print("\n[화면 2: 분석] 후보 경로별 신뢰도 점수 (점수 = 100 x 생존확률 x 정시도착확률)")
print(show.to_string(index=False))

# ---------------- 화면 3. 검토 (상태 뱃지·필터) ----------------
badge = {"신뢰": "완료(추천 가능)", "주의": "승인대기(통제관 검토)", "위험": "반려(재탐색)"}
table["상태"] = table["등급"].map(badge)
print("\n[화면 3: 검토] 상태별 필터")
for st in badge.values():
    print(f"  {st}: {table.loc[table['상태'] == st, '경로'].tolist()}")

# ---------------- 화면 4. 결과 ----------------
best = table.sort_values(["점수", "예상소요(P50)"], ascending=[False, True]).iloc[0]
print(f"\n[화면 4: 결과] 추천 {best['경로']} - 점수 {best['점수']}({best['등급']}), 생존확률 {best['생존확률']:.1%}, 정시도착 {best['정시도착']:.1%}")
print("핵심 화면 4종(입력-분석-검토-결과) 구현 완료: 4/4")

# ---------------- 예측 보정(Calibration) 검증 ----------------
# 서로 다른 목적지·기한의 가상 수송 60건: 예측 정시확률 vs 별도 난수로 모의한 '실제 운행 결과'
rng = np.random.default_rng(21)
pred, actual = [], []
for k in range(60):
    c = candidates(f"N{int(rng.integers(0, 6)):02d}07")[int(rng.integers(0, 2))] if k % 2 else candidates(f"N{int(rng.integers(0, 6)):02d}07")[0]
    dl = float(rng.uniform(60, 100))
    pred.append(score_route(c, dl, n=2000, seed=k)["정시도착"])
    actual.append(float(simulate_minutes(c["es"], 1, rng)[0] <= dl))
pred, actual = np.array(pred), np.array(actual)
brier = np.mean((pred - actual) ** 2)
print(f"\n정시도착 예측 보정 검증(가상 운행 60건): Brier 점수 {brier:.3f} (0에 가까울수록 좋음, 무정보 예측 0.25)")
for lo, hi in [(0, 0.5), (0.5, 0.9), (0.9, 1.01)]:
    m = (pred >= lo) & (pred < hi)
    if m.any():
        print(f"  예측 {lo:.0%}~{min(hi, 1):.0%} 구간 {m.sum():2d}건: 평균 예측 {pred[m].mean():.1%}, 실제 정시율 {actual[m].mean():.1%}")

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for e in edges:
    (x1, y1), (x2, y2) = nodes[e["u"]], nodes[e["v"]]
    ax[0, 0].plot([x1, x2], [y1, y2], color="lightgray", lw=1)
for t in threats:
    ax[0, 0].add_patch(plt.Circle((t["x"], t["y"]), t["r"], color="red", alpha=0.15))
for i, c in enumerate(cands):
    xy = np.array([nodes[n] for n in c["path"]]); ax[0, 0].plot(xy[:, 0], xy[:, 1], lw=2, label=f"R{i + 1}")
ax[0, 0].set_aspect("equal"); ax[0, 0].legend(fontsize=8); ax[0, 0].set_title("1. Input: request and candidate routes")
for i, c in enumerate(cands):
    ax[0, 1].hist(c["sims"], bins=40, alpha=0.5, label=f"R{i + 1}")
ax[0, 1].axvline(REQ["deadline_min"], color="k", ls="--"); ax[0, 1].set_xlabel("travel time (min)"); ax[0, 1].legend()
ax[0, 1].set_title("2. Analysis: Monte Carlo travel time")
cols = {"신뢰": "tab:green", "주의": "tab:orange", "위험": "tab:red"}
ax[1, 0].bar(table["경로"], table["점수"], color=[cols[g] for g in table["등급"]])
ax[1, 0].axhline(85, ls="--", c="gray"); ax[1, 0].axhline(60, ls=":", c="gray"); ax[1, 0].set_ylim(0, 100)
ax[1, 0].set_title("3. Review: score (green=trusted, orange=review, red=reject)")
ax[1, 1].plot([0, 1], [0, 1], "k--", lw=1)
bins = np.digitize(pred, [0.2, 0.4, 0.6, 0.8])
ax[1, 1].plot([pred[bins == b].mean() for b in np.unique(bins)], [actual[bins == b].mean() for b in np.unique(bins)], "o-")
ax[1, 1].set_xlabel("predicted on-time prob."); ax[1, 1].set_ylabel("observed on-time rate"); ax[1, 1].set_title("4. Result: calibration check")
plt.suptitle("MOV02 convoy reliability score engine (synthetic)"); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 05 예상 실행 결과 (Output)</b></summary>

```text
[화면 1: 입력] 수송 요청 {'req_id': 'REQ-0101', 'dest': 'N0307', 'cargo_t': 12.0, 'deadline_min': 80} | 가상 위협 구역 3 개

[화면 2: 분석] 후보 경로별 신뢰도 점수 (점수 = 100 x 생존확률 x 정시도착확률)
경로  구간 수  위협 통과   생존확률  정시도착 예상소요(P50) 예상소요(P90)   점수 등급
R1     9      3  57.8% 67.4%       77분       86분 38.9 위험
R2    10      0 100.0% 95.1%       69분       77분 95.1 신뢰
R3    10      0 100.0% 75.3%       75분       85분 75.3 주의
R4    10      0 100.0% 54.2%       79분       89분 54.2 위험

[화면 3: 검토] 상태별 필터
  완료(추천 가능): ['R2']
  승인대기(통제관 검토): ['R3']
  반려(재탐색): ['R1', 'R4']

[화면 4: 결과] 추천 R2 - 점수 95.1(신뢰), 생존확률 100.0%, 정시도착 95.1%
핵심 화면 4종(입력-분석-검토-결과) 구현 완료: 4/4

정시도착 예측 보정 검증(가상 운행 60건): Brier 점수 0.086 (0에 가까울수록 좋음, 무정보 예측 0.25)
  예측 0%~50% 구간 21건: 평균 예측 23.3%, 실제 정시율 19.0%
  예측 50%~90% 구간 14건: 평균 예측 72.6%, 실제 정시율 78.6%
  예측 90%~100% 구간 25건: 평균 예측 98.4%, 실제 정시율 100.0%
```
</details>
